### Подготовка окружения



In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import polars as pl
import bottleneck as bn
from IPython.display import display

# Файлы доступны и при запуске ноутбука из папки hw_5,
# и при запуске из корня репозитория python-homework.
def find_dataset(filename: str) -> Path:
    locations = [Path.cwd() / filename, Path.cwd() / "hw_5" / filename]
    for location in locations:
        if location.is_file():
            return location
    raise FileNotFoundError(
        f"Не найден {filename}. Поместите оба CSV-файла рядом с ноутбуком "
        "в папку hw_5. Текущая папка: " + str(Path.cwd())
    )

train_path = find_dataset("train.csv")
housing_path = find_dataset("Housing.csv")
print("Titanic:", train_path)
print("Housing:", housing_path)
print("Версия Polars:", pl.__version__)
print("Версия Pandas:", pd.__version__)
print("Версия Bottleneck:", bn.__version__)


Titanic: c:\Users\roman\Desktop\python-homework\hw_5\train.csv
Housing: c:\Users\roman\Desktop\python-homework\hw_5\Housing.csv
Версия Polars: 2.0.0
Версия Pandas: 3.0.6
Версия Bottleneck: 1.6.0


---
# Часть 1. Polars

### 1. Чтение `train.csv` средствами Polars

In [2]:
titanic_pl = pl.read_csv(train_path)
print(f"Размер таблицы: {titanic_pl.height} строк × {titanic_pl.width} столбцов")
display(titanic_pl.head(10))


Размер таблицы: 891 строк × 12 столбцов


PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
i64,i64,i64,str,str,f64,i64,i64,str,f64,str,str
1,0,3,"""Braund, Mr. Owen Harris""","""male""",22.0,1,0,"""A/5 21171""",7.25,null,"""S"""
2,1,1,"""Cumings, Mrs. John Bradley (Fl…","""female""",38.0,1,0,"""PC 17599""",71.2833,"""C85""","""C"""
3,1,3,"""Heikkinen, Miss. Laina""","""female""",26.0,0,0,"""STON/O2. 3101282""",7.925,null,"""S"""
4,1,1,"""Futrelle, Mrs. Jacques Heath (…","""female""",35.0,1,0,"""113803""",53.1,"""C123""","""S"""
5,0,3,"""Allen, Mr. William Henry""","""male""",35.0,0,0,"""373450""",8.05,null,"""S"""
6,0,3,"""Moran, Mr. James""","""male""",null,0,0,"""330877""",8.4583,null,"""Q"""
7,0,1,"""McCarthy, Mr. Timothy J""","""male""",54.0,0,0,"""17463""",51.8625,"""E46""","""S"""
8,0,3,"""Palsson, Master. Gosta Leonard""","""male""",2.0,3,1,"""349909""",21.075,null,"""S"""
9,1,3,"""Johnson, Mrs. Oscar W (Elisabe…","""female""",27.0,0,2,"""347742""",11.1333,null,"""S"""


### 2. Общая информация: типы данных, пропуски и статистические показатели



In [3]:
null_values = titanic_pl.null_count().row(0)
column_info = pd.DataFrame({
    "Столбец": titanic_pl.columns,
    "Тип Polars": [str(dtype) for dtype in titanic_pl.dtypes],
    "Пропуски": null_values,
})
print("Типы данных и количество пропусков:")
display(column_info)

print("Описательная статистика:")
display(titanic_pl.describe())

print("Средние значения выбранных числовых показателей:")
display(titanic_pl.select(
    pl.col("Age").mean().alias("Средний возраст"),
    pl.col("Fare").mean().alias("Средняя цена билета"),
    pl.col("Survived").mean().alias("Доля выживших"),
))


Типы данных и количество пропусков:


,Столбец,Тип Polars,Пропуски
0,PassengerId,Int64,0
1,Survived,Int64,0
2,Pclass,Int64,0
3,Name,String,0
4,Sex,String,0
5,Age,Float64,177
6,SibSp,Int64,0
7,Parch,Int64,0
8,Ticket,String,0
9,Fare,Float64,0


Описательная статистика:


statistic,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
str,f64,f64,f64,str,str,f64,f64,f64,str,f64,str,str
"""count""",891.0,891.0,891.0,"""891""","""891""",714.0,891.0,891.0,"""891""",891.0,"""204""","""889"""
"""null_count""",0.0,0.0,0.0,"""0""","""0""",177.0,0.0,0.0,"""0""",0.0,"""687""","""2"""
"""mean""",446.0,0.383838,2.308642,null,null,29.699118,0.523008,0.381594,null,32.204208,null,null
"""std""",257.353842,0.486592,0.836071,null,null,14.526497,1.102743,0.806057,null,49.693429,null,null
"""min""",1.0,0.0,1.0,"""Abbing, Mr. Anthony""","""female""",0.42,0.0,0.0,"""110152""",0.0,"""A10""","""C"""
"""25%""",224.0,0.0,2.0,null,null,20.0,0.0,0.0,null,7.925,null,null
"""50%""",446.0,0.0,3.0,null,null,28.0,0.0,0.0,null,14.4542,null,null
"""75%""",669.0,1.0,3.0,null,null,38.0,1.0,0.0,null,31.0,null,null
"""max""",891.0,1.0,3.0,"""van Melkebeke, Mr. Philemon""","""male""",80.0,8.0,6.0,"""WE/P 5735""",512.3292,"""T""","""S"""


Средние значения выбранных числовых показателей:


Средний возраст,Средняя цена билета,Доля выживших
f64,f64,f64
29.699118,32.204208,0.383838


### 3. Количество пассажиров в каждом классе (`Pclass`)



In [4]:
pclass_counts = titanic_pl.get_column("Pclass").value_counts().sort("Pclass")
print("Количество пассажиров по классам:")
display(pclass_counts)
assert int(pclass_counts["count"].sum()) == titanic_pl.height


Количество пассажиров по классам:


Pclass,count
i64,u32
1,216
2,184
3,491


### 4. Число выживших мужчин и женщин



In [5]:
survival_by_sex = (
    titanic_pl
    .group_by("Sex")
    .agg(
        pl.len().alias("Всего пассажиров"),
        pl.col("Survived").sum().alias("Выживших"),
    )
    .with_columns(
        (pl.col("Выживших") / pl.col("Всего пассажиров") * 100)
        .round(2)
        .alias("Выживаемость, %")
    )
    .sort("Sex")
)
display(survival_by_sex)
assert int(survival_by_sex["Выживших"].sum()) == int(titanic_pl["Survived"].sum())


Sex,Всего пассажиров,Выживших,"Выживаемость, %"
str,u32,i64,f64
"""female""",314,233,74.2
"""male""",577,109,18.89


### 5. Пассажиры старше 44 лет



In [6]:
older_than_44 = titanic_pl.filter(pl.col("Age") > 44)
print(f"Всего пассажиров старше 44 лет: {older_than_44.height}")
display(older_than_44.select([
    "PassengerId", "Name", "Sex", "Age", "Pclass", "Survived"
]).head(15))


Всего пассажиров старше 44 лет: 115


PassengerId,Name,Sex,Age,Pclass,Survived
i64,str,str,f64,i64,i64
7,"""McCarthy, Mr. Timothy J""","""male""",54.0,1,0
12,"""Bonnell, Miss. Elizabeth""","""female""",58.0,1,1
16,"""Hewlett, Mrs. (Mary D Kingcome…","""female""",55.0,2,1
34,"""Wheadon, Mr. Edward H""","""male""",66.0,2,0
53,"""Harper, Mrs. Henry Sleeper (My…","""female""",49.0,1,1
…,…,…,…,…,…
111,"""Porter, Mr. Walter Chamberlain""","""male""",47.0,1,0
117,"""Connors, Mr. Patrick""","""male""",70.5,3,0
125,"""White, Mr. Percival Wayland""","""male""",54.0,1,0


---
# Часть 2. Ускорение вычислений Pandas

### 1. Чтение Titanic через Pandas

In [7]:
titanic_pd = pd.read_csv(train_path)
print(f"Размер таблицы: {titanic_pd.shape[0]} строк × {titanic_pd.shape[1]} столбцов")
display(titanic_pd.head())


Размер таблицы: 891 строк × 12 столбцов


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


### 2. Средний возраст и стандартное отклонение через Bottleneck



In [8]:
ages = titanic_pd["Age"].to_numpy(dtype=float)
mean_age = bn.nanmean(ages)
std_age = bn.nanstd(ages, ddof=1)

print(f"Средний возраст (Bottleneck): {mean_age:.4f}")
print(f"Стандартное отклонение (Bottleneck, ddof=1): {std_age:.4f}")
print(f"Пассажиров с известным возрастом: {np.count_nonzero(~np.isnan(ages))}")

# Проверка совпадения вычислений с Pandas.
assert np.isclose(mean_age, titanic_pd["Age"].mean())
assert np.isclose(std_age, titanic_pd["Age"].std())


Средний возраст (Bottleneck): 29.6991
Стандартное отклонение (Bottleneck, ddof=1): 14.5265
Пассажиров с известным возрастом: 714


### 3. Новый столбец `Fare_new`



In [9]:
titanic_pd["Fare_new"] = titanic_pd["Fare"].apply(lambda fare: fare * 1.3)
print("Стоимость билета до и после увеличения на 30%:")
display(titanic_pd[["PassengerId", "Name", "Fare", "Fare_new"]].head(12))

# Контроль правильности для всех пассажиров, включая возможные пропуски.
assert np.allclose(
    titanic_pd["Fare_new"].to_numpy(dtype=float),
    (titanic_pd["Fare"] * 1.3).to_numpy(dtype=float),
    equal_nan=True,
)


Стоимость билета до и после увеличения на 30%:


,PassengerId,Name,Fare,Fare_new
0,1,"Braund, Mr. Owen Harris",7.2500,9.42500
1,2,"Cumings, Mrs. John Bradley (Florence Briggs Th...",71.2833,92.66829
2,3,"Heikkinen, Miss. Laina",7.9250,10.30250
3,4,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",53.1000,69.03000
4,5,"Allen, Mr. William Henry",8.0500,10.46500
5,6,"Moran, Mr. James",8.4583,10.99579
6,7,"McCarthy, Mr. Timothy J",51.8625,67.42125
7,8,"Palsson, Master. Gosta Leonard",21.0750,27.39750
8,9,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",11.1333,14.47329
9,10,"Nasser, Mrs. Nicholas (Adele Achem)",30.0708,39.09204


---
# Часть 3. Оптимизация типов данных Pandas

### 1. Чтение `Housing.csv`



In [10]:
housing = pd.read_csv(housing_path)
print(f"Размер таблицы: {housing.shape[0]} строк × {housing.shape[1]} столбцов")
display(housing.head())

print("Начальные типы и число уникальных значений:")
display(pd.DataFrame({
    "Тип до оптимизации": housing.dtypes.astype(str),
    "Число уникальных значений": housing.nunique(dropna=False),
    "Пропуски": housing.isna().sum(),
}))

before_bytes = int(housing.memory_usage(deep=True).sum())
print(f"Общий объём памяти до оптимизации: {before_bytes:,} байт")


Размер таблицы: 545 строк × 13 столбцов


,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished


Начальные типы и число уникальных значений:


,Тип до оптимизации,Число уникальных значений,Пропуски
price,int64,219,0
area,int64,284,0
bedrooms,int64,6,0
bathrooms,int64,4,0
stories,int64,4,0
mainroad,str,2,0
guestroom,str,2,0
basement,str,2,0
hotwaterheating,str,2,0
airconditioning,str,2,0


Общий объём памяти до оптимизации: 227,244 байт


### 2. Определение подходящего типа для каждого столбца

Выбор определяется **реальными диапазонами значений** и смыслом полей, без округления или потери информации.

| Столбец | Новый тип | Обоснование |
|---|---|---|
| `price` | `uint32` | Цена — неотрицательное целое; её диапазон помещается в 32 бита. |
| `area` | `uint16` | Площадь — неотрицательное целое, для которого достаточно 16 бит. |
| `bedrooms` | `uint8` | Небольшое число спален, достаточно 8 бит. |
| `bathrooms` | `uint8` | Небольшое число ванных комнат. |
| `stories` | `uint8` | Небольшое число этажей. |
| `mainroad` | `bool` | Значения `yes`/`no` представлены как True/False. |
| `guestroom` | `bool` | Двоичный признак наличия гостевой комнаты. |
| `basement` | `bool` | Двоичный признак наличия подвала. |
| `hotwaterheating` | `bool` | Двоичный признак горячего водоснабжения/отопления. |
| `airconditioning` | `bool` | Двоичный признак кондиционирования. |
| `parking` | `uint8` | Небольшое число парковочных мест. |
| `prefarea` | `bool` | Двоичный признак предпочтительного района. |
| `furnishingstatus` | `category` | Повторяющиеся категории мебели; экономнее, чем строки `object`. |




In [11]:
housing_optimized = housing.copy()

# Целочисленные столбцы: наименьший подходящий беззнаковый тип.
# price → uint32; area → uint16;
# bedrooms, bathrooms, stories, parking → uint8.
integer_columns = ["price", "area", "bedrooms", "bathrooms", "stories", "parking"]
for column in integer_columns:
    if housing_optimized[column].isna().any() or (housing_optimized[column] < 0).any():
        raise ValueError(f"{column}: нельзя безопасно преобразовать в беззнаковый тип")
    housing_optimized[column] = pd.to_numeric(
        housing_optimized[column], downcast="unsigned"
    )

# Бинарные столбцы: true/false.
# mainroad, guestroom, basement, hotwaterheating, airconditioning, prefarea → bool.
binary_columns = [
    "mainroad", "guestroom", "basement",
    "hotwaterheating", "airconditioning", "prefarea",
]
for column in binary_columns:
    if not housing_optimized[column].isin(["yes", "no"]).all():
        raise ValueError(f"{column}: обнаружены значения, отличные от yes/no")
    housing_optimized[column] = (
        housing_optimized[column].map({"yes": True, "no": False}).astype(bool)
    )

# furnishingstatus → category: компактное хранение 3 повторяющихся категорий.
housing_optimized["furnishingstatus"] = housing_optimized["furnishingstatus"].astype("category")

print("Типы после оптимизации:")
display(housing_optimized.dtypes.to_frame("Тип после оптимизации"))
display(housing_optimized.head())


Типы после оптимизации:


,Тип после оптимизации
price,uint32
area,uint16
bedrooms,uint8
bathrooms,uint8
stories,uint8
mainroad,bool
guestroom,bool
basement,bool
hotwaterheating,bool
airconditioning,bool


,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,True,False,False,False,True,2,True,furnished
1,12250000,8960,4,4,4,True,False,False,False,True,3,False,furnished
2,12250000,9960,3,2,2,True,False,True,False,False,2,True,semi-furnished
3,12215000,7500,4,2,2,True,False,True,False,True,3,True,furnished
4,11410000,7420,4,1,2,True,True,True,False,True,2,False,furnished


### 3. Сравнение потребления памяти до и после оптимизации



In [12]:
memory_before = housing.memory_usage(index=False, deep=True)
memory_after = housing_optimized.memory_usage(index=False, deep=True)

memory_report = pd.DataFrame({
    "Тип до": housing.dtypes.astype(str),
    "Тип после": housing_optimized.dtypes.astype(str),
    "До, байт": memory_before,
    "После, байт": memory_after,
})
memory_report["Экономия, %"] = (
    (1 - memory_report["После, байт"] / memory_report["До, байт"]) * 100
).round(2)
display(memory_report)

after_bytes = int(housing_optimized.memory_usage(deep=True).sum())
saved_bytes = before_bytes - after_bytes
saved_percent = 100 * saved_bytes / before_bytes
print(f"Память до оптимизации:    {before_bytes:,} байт")
print(f"Память после оптимизации: {after_bytes:,} байт")
print(f"Сэкономлено:             {saved_bytes:,} байт ({saved_percent:.2f}%)")

# Числовые значения должны сохраняться точно, без изменения величин.
for column in integer_columns:
    assert np.array_equal(
        housing_optimized[column].to_numpy(), housing[column].to_numpy()
    ), f"Изменились значения столбца {column}"

# Двоичные логические признаки можно однозначно преобразовать обратно в yes/no.
for column in binary_columns:
    restored = housing_optimized[column].map({True: "yes", False: "no"})
    assert np.array_equal(restored.to_numpy(), housing[column].to_numpy()), column

# Категория сохраняет те же строковые значения.
assert np.array_equal(
    housing_optimized["furnishingstatus"].astype(str).to_numpy(),
    housing["furnishingstatus"].to_numpy(),
)
assert after_bytes < before_bytes
print("Проверка: все исходные значения сохранены, память сокращена.")


,Тип до,Тип после,"До, байт","После, байт","Экономия, %"
price,int64,uint32,4360,2180,50.00
area,int64,uint16,4360,1090,75.00
bedrooms,int64,uint8,4360,545,87.50
bathrooms,int64,uint8,4360,545,87.50
stories,int64,uint8,4360,545,87.50
mainroad,str,bool,28263,545,98.07
guestroom,str,bool,27892,545,98.05
basement,str,bool,27986,545,98.05
hotwaterheating,str,bool,27820,545,98.04
airconditioning,str,bool,27967,545,98.05


Память до оптимизации:    227,244 байт
Память после оптимизации: 9,578 байт
Сэкономлено:             217,666 байт (95.79%)
Проверка: все исходные значения сохранены, память сокращена.
